# MURA → BTXRD transfer learning

End-to-end notebook:

1. **Stage 1 — MURA:** Fine-tune an ImageNet **ResNet50V2** backbone with a dense head for **normal vs abnormal** (study-level labels mapped to every image in each study).
2. **Stage 2 — BTXRD (Figshare):** Load the MURA-trained weights into the **same backbone**, attach a **fresh classification head**, and train **tumor vs non-tumor** using `dataset.xlsx` (`tumor` column: 1 = tumor present, 0 = normal / no tumor).

**Prerequisites:** `data/MURA-v1.1/` and `data/BTXRD/` (with `images/` and `dataset.xlsx`). Install deps: `pip install -r requirements.txt` (includes `openpyxl` for Excel).

## 0. Imports and reproducibility

In [1]:
import os
import random
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

TensorFlow: 2.21.0
GPUs: []


## 1. Paths and hyperparameters

In [2]:
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data").is_dir():
    REPO_ROOT = Path.cwd().resolve().parent
DATA_ROOT = REPO_ROOT / "data"
MURA_ROOT = DATA_ROOT / "MURA-v1.1"
BTXRD_ROOT = DATA_ROOT / "BTXRD"
BTXRD_IMAGES = BTXRD_ROOT / "images"
BTXRD_XLSX = BTXRD_ROOT / "dataset.xlsx"
CKPT_DIR = REPO_ROOT / "models" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
MURA_MODEL_PATH = CKPT_DIR / "mura_binary.keras"
BTXRD_MODEL_PATH = CKPT_DIR / "btxrd_from_mura.keras"

IMG_SIZE = 224
BATCH_SIZE = 32
MURA_EPOCHS = 5
BTXRD_EPOCHS = 8
BTXRD_LR = 1e-4

assert MURA_ROOT.is_dir(), f"Missing {MURA_ROOT}"
assert BTXRD_IMAGES.is_dir(), f"Missing {BTXRD_IMAGES}"
assert BTXRD_XLSX.is_file(), f"Missing {BTXRD_XLSX}"
print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /Users/adyan/Desktop/COSC 4337/Bone-Cancer-Detection/data


## 2. MURA — join image paths to study labels

In [3]:
def load_mura_frame(data_root: Path, split: str):
    mura = data_root / "MURA-v1.1"
    studies = pd.read_csv(
        mura / f"{split}_labeled_studies.csv",
        header=None,
        names=["study_dir", "label"],
        dtype={"study_dir": str, "label": int},
    )
    images = pd.read_csv(
        mura / f"{split}_image_paths.csv",
        header=None,
        names=["rel_path"],
        dtype=str,
    )
    images["study_dir"] = images["rel_path"].apply(
        lambda p: f"{Path(p).parent.as_posix()}/"
    )
    merged = images.merge(studies, on="study_dir", how="inner")
    paths = (data_root / merged["rel_path"].str.strip()).map(lambda p: p.as_posix())
    labels = merged["label"].astype("float32")
    return paths, labels

train_paths, train_y = load_mura_frame(DATA_ROOT, "train")
val_paths, val_y = load_mura_frame(DATA_ROOT, "valid")
print("MURA train images:", len(train_paths), "| valid:", len(val_paths))

MURA train images: 36808 | valid: 3197


## 3. MURA — `tf.data` + ResNet50V2 preprocessing

In [4]:
def decode_image(path, label, size):
    data = tf.io.read_file(path)
    img = tf.io.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [size, size])
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.resnet_v2.preprocess_input(img)
    return img, label


def make_dataset(paths, labels, *, size, batch_size, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 10_000), reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, y: decode_image(p, y, size),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

mura_train_ds = make_dataset(
    train_paths.tolist(), train_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
mura_val_ds = make_dataset(
    val_paths.tolist(), val_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

## 4. MURA — model (ImageNet backbone + head)

The backbone is named **`radiograph_backbone`** so we can copy its weights into the BTXRD model after MURA training.

In [5]:
def build_mura_model(img_size: int) -> tf.keras.Model:
    backbone = tf.keras.applications.ResNet50V2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights="imagenet",
        name="radiograph_backbone",
    )
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(img_size, img_size, 3), name="image")
    x = backbone(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.BatchNormalization(name="mura_bn")(x)
    x = tf.keras.layers.Dense(512, activation="relu", name="mura_fc1")(x)
    x = tf.keras.layers.Dropout(0.5, name="mura_drop1")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="mura_fc2")(x)
    x = tf.keras.layers.Dropout(0.35, name="mura_drop2")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="mura_abnormal_prob")(x)
    model = tf.keras.Model(inputs, outputs, name="mura_resnet50v2")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model

mura_model = build_mura_model(IMG_SIZE)
mura_model.summary()

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "mura_resnet50v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ radiograph_backbone             │ (None, 7, 7, 2048)     │    23,564,800 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_bn (BatchNormalization)    │ (None, 2048)           │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_fc1 (Dense)                │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_drop1 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_fc2 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_drop2 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mura_abnormal_prob (Dense)      │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,753,665 (94.43 MB)

 Trainable params: 1,184,769 (4.52 MB)

 Non-trainable params: 23,568,896 (89.91 MB)

## 5. MURA — train and save checkpoint

In [ ]:
history_mura = mura_model.fit(
    mura_train_ds,
    validation_data=mura_val_ds,
    epochs=MURA_EPOCHS,
)
mura_model.save(MURA_MODEL_PATH)
print("Saved:", MURA_MODEL_PATH)

Epoch 1/5
 349/1151 ━━━━━━━━━━━━━━━━━━━━ 8:22 626ms/step - accuracy: 0.5876 - auc: 0.6190 - loss: 0.7551

## 6. BTXRD — build image list and binary `tumor` labels

In [ ]:
def resolve_image_path(images_dir: Path, image_id: str) -> Optional[Path]:
    p = images_dir / str(image_id).strip()
    if p.is_file():
        return p
    stem = Path(str(image_id)).stem
    for ext in (".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"):
        c = images_dir / f"{stem}{ext}"
        if c.is_file():
            return c
    return None

btxrd_df = pd.read_excel(BTXRD_XLSX, engine="openpyxl")
btxrd_df.columns = [str(c).strip() for c in btxrd_df.columns]
if "tumor" not in btxrd_df.columns:
    raise KeyError(f"Expected a 'tumor' column; got: {list(btxrd_df.columns)}")

rows = []
for _, r in btxrd_df.iterrows():
    img_id = r["image_id"]
    p = resolve_image_path(BTXRD_IMAGES, img_id)
    if p is None:
        continue
    rows.append((p.as_posix(), int(r["tumor"])))

btx_paths = [a for a, _ in rows]
btx_labels = np.array([b for _, b in rows], dtype="float32")
print("BTXRD samples with files on disk:", len(btx_paths))
print("Tumor rate:", float(btx_labels.mean()))

b_train_p, b_val_p, b_train_y, b_val_y = train_test_split(
    btx_paths, btx_labels, test_size=0.2, random_state=SEED, stratify=btx_labels
)
print("BTXRD train / val:", len(b_train_p), len(b_val_p))

## 7. BTXRD — datasets (same preprocessing as MURA)

In [ ]:
btx_train_ds = make_dataset(
    b_train_p, b_train_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
btx_val_ds = make_dataset(
    b_val_p, b_val_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

## 8. BTXRD — new model with MURA-trained backbone

Load the saved MURA model, instantiate a **new** model with the same backbone shape, **`set_weights`** on `radiograph_backbone` from MURA, add a **new head** for tumor classification, optionally unfreeze the top layers of the backbone for fine-tuning.

In [ ]:
def build_btxrd_model(img_size: int, backbone_trainable_layers: int = 30) -> tf.keras.Model:
    backbone = tf.keras.applications.ResNet50V2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights=None,
        name="radiograph_backbone",
    )
    backbone.trainable = True
    if backbone_trainable_layers > 0:
        for layer in backbone.layers[:-backbone_trainable_layers]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=(img_size, img_size, 3), name="image")
    x = backbone(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.BatchNormalization(name="btx_bn")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="btx_fc1")(x)
    x = tf.keras.layers.Dropout(0.45, name="btx_drop1")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="btx_tumor_prob")(x)
    model = tf.keras.Model(inputs, outputs, name="btxrd_resnet50v2_from_mura")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(BTXRD_LR),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model


mura_loaded = tf.keras.models.load_model(MURA_MODEL_PATH, compile=False)
btxrd_model = build_btxrd_model(IMG_SIZE)
src_bb = mura_loaded.get_layer("radiograph_backbone")
dst_bb = btxrd_model.get_layer("radiograph_backbone")
dst_bb.set_weights(src_bb.get_weights())
print("Copied backbone weights: MURA → BTXRD")

btxrd_model.summary()

## 9. BTXRD — train and save

In [ ]:
history_btx = btxrd_model.fit(
    btx_train_ds,
    validation_data=btx_val_ds,
    epochs=BTXRD_EPOCHS,
)
btxrd_model.save(BTXRD_MODEL_PATH)
print("Saved:", BTXRD_MODEL_PATH)

## 10. (Optional) Quick learning curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history_mura.history.get("loss", []), label="train")
ax[0].plot(history_mura.history.get("val_loss", []), label="val")
ax[0].set_title("MURA loss")
ax[0].legend()

ax[1].plot(history_btx.history.get("loss", []), label="train")
ax[1].plot(history_btx.history.get("val_loss", []), label="val")
ax[1].set_title("BTXRD loss")
ax[1].legend()
plt.tight_layout()
plt.show()